# Central Bank Boards

Notebook limpio para extraer miembros de board desde una lista de paises y sitios oficiales.

Idea:
- tu le pasas `country` y `website`
- el agente busca solo dentro de ese sitio
- si ya conoces la pagina exacta del board, tambien puedes pasar `board_page_url`
- el resultado guarda `source_url` para poder revisar

In [ ]:
!pip install ddgs requests beautifulsoup4 lxml pandas -q

In [ ]:
import os
from getpass import getpass

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass("Mistral API key: ")

In [ ]:
from ddgs import DDGS
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import os
import re

API_KEY = os.environ["MISTRAL_API_KEY"]
HEADERS = {"User-Agent": "Mozilla/5.0"}
BOARD_HINTS = [
    "board",
    "leadership",
    "management",
    "governors",
    "directors",
    "governance",
    "executive board",
    "consejo",
    "directorio",
    "junta",
]

In [ ]:
def same_site(base_url, candidate_url):
    base_host = urlparse(base_url).netloc.lower().replace("www.", "")
    candidate_host = urlparse(candidate_url).netloc.lower().replace("www.", "")
    return base_host == candidate_host


def clean_text(text):
    text = re.sub(r"\s+", " ", text or "")
    return text.strip()


def looks_like_name(text):
    text = clean_text(text)
    if len(text.split()) < 2 or len(text.split()) > 5:
        return False
    if any(ch.isdigit() for ch in text):
        return False
    blocked = {
        "the governing board",
        "board of directors",
        "executive board",
        "annual report",
        "the president",
        "the chairman",
    }
    lowered = text.lower()
    if lowered in blocked:
        return False
    return bool(re.match(r"^[A-ZÁÉÍÓÚÑ][A-Za-zÁÉÍÓÚÑáéíóúñ'`.-]+(?:\s+[A-ZÁÉÍÓÚÑ][A-Za-zÁÉÍÓÚÑáéíóúñ'`.-]+){1,4}$", text))


def llm_extract(text, country):
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
    }
    prompt = f"""\nExtract current central bank board or leadership members for {country}.\n\nRules:\n- Only return real people.\n- Exclude generic phrases like 'The Governing Board' or 'Annual Report'.\n- Exclude commentary, notes, and explanations.\n- Prefer current members shown on the page.\n- If no people are clearly listed, return nothing.\n\nReturn one line per person in this exact format:\nName - Role\n\nTEXT:\n{text}\n"""
    data = {
        "model": "mistral-small",
        "messages": [
            {"role": "system", "content": "You extract structured people data from official institutional webpages."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0,
    }
    response = requests.post(url, json=data, headers=headers, timeout=60)
    if response.status_code != 200:
        return ""
    return response.json()["choices"][0]["message"]["content"]


In [ ]:
def search_board_pages(country, website, max_results=8):
    queries = [
        f"site:{urlparse(website).netloc} {country} central bank board",
        f"site:{urlparse(website).netloc} leadership",
        f"site:{urlparse(website).netloc} governors",
        f"site:{urlparse(website).netloc} directors",
        f"site:{urlparse(website).netloc} management",
    ]

    found = []
    seen = set()
    with DDGS() as ddgs:
        for query in queries:
            for result in ddgs.text(query, max_results=max_results):
                href = result.get("href", "")
                if not href.startswith("http"):
                    continue
                if not same_site(website, href):
                    continue
                haystack = (href + " " + result.get("title", "") + " " + result.get("body", "")).lower()
                if not any(hint in haystack for hint in BOARD_HINTS):
                    continue
                if href in seen:
                    continue
                seen.add(href)
                found.append(href)
    return found[:8]


def find_board_page(country, website):
    candidates = search_board_pages(country, website)
    if candidates:
        return candidates[0]

    try:
        response = requests.get(website, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(response.text, "lxml")
        for link in soup.find_all("a", href=True):
            href = urljoin(website, link["href"])
            text = clean_text(link.get_text(" ", strip=True)).lower()
            combined = (href + " " + text).lower()
            if same_site(website, href) and any(hint in combined for hint in BOARD_HINTS):
                return href
    except Exception:
        pass

    return website


def get_clean_text(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
        soup = BeautifulSoup(response.text, "lxml")

        for tag in soup(["script", "style", "noscript", "svg", "footer", "nav", "header"]):
            tag.decompose()

        texts = []
        for tag in soup.find_all(["h1", "h2", "h3", "li", "p", "td", "span"]):
            text = clean_text(tag.get_text(" ", strip=True))
            lowered = text.lower()
            if len(text) < 4:
                continue
            if any(x in lowered for x in ["cookie", "privacy", "subscribe", "newsletter", "all rights reserved"]):
                continue
            texts.append(text)

        return "\n".join(texts[:500])
    except Exception:
        return ""


def parse_people(text, country, source_url):
    rows = []
    seen = set()
    for line in text.split("\n"):
        if "-" not in line:
            continue
        name, role = line.split("-", 1)
        name = clean_text(name).strip("*• ")
        role = clean_text(role)
        if not looks_like_name(name):
            continue
        if len(role) < 3:
            continue
        key = (country, name, role)
        if key in seen:
            continue
        seen.add(key)
        rows.append({
            "country": country,
            "name": name,
            "role": role,
            "source_url": source_url,
        })
    return rows


In [ ]:
def agent(country, website, board_page_url=""):
    source_url = board_page_url or find_board_page(country, website)
    text = get_clean_text(source_url)
    if len(text) < 200 and source_url != website:
        text = get_clean_text(website)
        source_url = website
    extracted = llm_extract(text, country)
    return parse_people(extracted, country, source_url)


def build_dataset(input_rows):
    all_rows = []
    for row in input_rows:
        country = row["country"]
        website = row["website"]
        board_page_url = row.get("board_page_url", "")
        print(f"Processing {country} -> {website}")
        try:
            rows = agent(country=country, website=website, board_page_url=board_page_url)
            all_rows.extend(rows)
        except Exception as exc:
            print("Error:", country, exc)
    df = pd.DataFrame(all_rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["country", "name", "role", "source_url"]).reset_index(drop=True)
    return df


## Lista de entrada

Aqui es donde pasas exactamente lo que pediste: pais y pagina web.

Si ya conoces la pagina del board, llenas `board_page_url` y el agente va directo ahi.

In [ ]:
input_rows = [
    {
        "country": "Chile",
        "website": "https://www.bcentral.cl",
        "board_page_url": "",
    },
    {
        "country": "Argentina",
        "website": "https://www.bcra.gob.ar",
        "board_page_url": "",
    },
    {
        "country": "Brazil",
        "website": "https://www.bcb.gov.br",
        "board_page_url": "",
    },
    {
        "country": "United States",
        "website": "https://www.federalreserve.gov",
        "board_page_url": "",
    },
    {
        "country": "European Central Bank",
        "website": "https://www.ecb.europa.eu",
        "board_page_url": "",
    },
]

input_df = pd.DataFrame(input_rows)
input_df

In [ ]:
df = build_dataset(input_rows)
df

In [ ]:
df.to_csv("central_bank_boards_from_websites.csv", index=False)